In [ ]:
import pandas as pd
import matplotlib as mpl
from tcr_toolbox.utils.plot_utils import set_mpl_params,plot_count_histogram
mpl, mylines = set_mpl_params(mpl=mpl)
from tcr_toolbox.sequencing_analysis_PRIVATE.plate_tcr_epi_count_analysis import init_umi_count_analysis_dir, read_umi_count_tsv, filter_UMI_counts_df_dict, plot_detected_tcr_epi_plate_map, write_tcr_epi_pair_well_count_files, read_gdna_counts_csv, read_tcr_epi_pair_counts_csv, calculate_tcr_epi_pair_exp_prob_matrix, obs_tcr_epi_count_binom_sf_test, add_reactive_epi_and_tcr_co_occurrence_bool_col, add_tcr_epi_pairs_only_supported_by_co_occurrence_bool_col, plot_tcr_epi_pair_ma_plot, plot_pvalue_corrected_rank_vs_pvalue_corrected, write_reactive_pair_results, eval_performance_run, add_pair_entropy_over_plate_col
from tcr_toolbox.utils.stat_utils import normalize_counts_df_by_total_counts
from tcr_toolbox.sequencing_analysis.utils import overlap_between_ref_and_count_names
import numpy as np
import os
from pathlib import Path

In [ ]:
project_dir = Path('/data/mel-pt-a_T200_vs_Ag200_screen-1')

In [ ]:
init_umi_count_analysis_dir(project_dir)

## Preprocessing

In [3]:
counts_df_dict = read_umi_count_tsv(project_dir = project_dir,
                                    plate_name_split_location = 2
                                    )

Reading plate: 1-3
Reading plate: 1-11
Reading plate: 1-9
Reading plate: 1-5
Reading plate: 1-8
Reading plate: 1-4
Reading plate: 1-2
Reading plate: 1-7
Reading plate: 1-1
Reading plate: 1-6
Reading plate: 1-13
Reading plate: 1-10
Reading plate: 1-12


In [ ]:
epi_umi_threshold_dict = {
    "1-3": 250, 
    "1-11": 200, 
    "1-9": 150,
    "1-5": 150,
    "1-8": 200,
    "1-4": 200, 
    "1-2": 200,
    "1-7": 150,
    "1-1": 200,
    "1-6": 200, 
    "1-13": 150, 
    "1-10": 150,
    "1-12": 175,
}


tcr_umi_threshold_dict = {
    "1-3": 150, 
    "1-11": 100, 
    "1-9": 75,
    "1-5": 100,
    "1-8": 60,
    "1-4": 200,
    "1-2": 200,
    "1-7": 60,
    "1-1": 100,
    "1-6": 100,
    "1-13": 75,
    "1-10": 70, 
    "1-12": 75, 
}

In [ ]:
counts_df_dict = filter_UMI_counts_df_dict(
    project_dir = project_dir, 
    counts_df_dict = counts_df_dict, 
    epi_umi_threshold=epi_umi_threshold_dict, 
    tcr_umi_threshold=tcr_umi_threshold_dict,
    min_start_fraction_of_top_umi_count=0.72, 
    min_end_fraction_of_top_umi_count = 0.12,
    num_bins=50
)

Filtering plate: 1-3
Number of unique epis: 150
Number of unique tcrs: 69
Filtering plate: 1-11
Number of unique epis: 189
Number of unique tcrs: 99
Filtering plate: 1-9
Number of unique epis: 194
Number of unique tcrs: 100
Filtering plate: 1-5
Number of unique epis: 179
Number of unique tcrs: 72
Filtering plate: 1-8
Number of unique epis: 192
Number of unique tcrs: 116
Filtering plate: 1-4
Number of unique epis: 100
Number of unique tcrs: 70
Filtering plate: 1-2
Number of unique epis: 98
Number of unique tcrs: 66
Filtering plate: 1-7
Number of unique epis: 181
Number of unique tcrs: 96
Filtering plate: 1-1
Number of unique epis: 126
Number of unique tcrs: 55
Filtering plate: 1-6
Number of unique epis: 124
Number of unique tcrs: 95
Filtering plate: 1-13
Number of unique epis: 173
Number of unique tcrs: 84
Filtering plate: 1-10
Number of unique epis: 228
Number of unique tcrs: 116
Filtering plate: 1-12
Number of unique epis: 223
Number of unique tcrs: 109


In [ ]:
# After individual TCR-Ag pair validation co-cultures, code can be re-run with a validating_pair_dict to color annotate dots in results plots. 
validating_pair_dict = {'epi_mel-pt-a_CCSER2_1384-tcr_9_17': True,
                               'epi_mel-pt-a_GFPT2_896-tcr_27_41': True,
                               'epi_mel-pt-a_CCSER2_1384-tcr_44_80': True, 
                               'epi_mel-pt-a_CCSER2_1384-tcr_76_150': True, 
                               'epi_mel-pt-a_CCSER2_1384-tcr_96_188': True, 

                               'epi_mel-pt-a_CCSER2_1384-tcr_5_7': True, 
                               'epi_mel-pt-a_CCSER2_1384-tcr_40_76': True, 
                               'epi_mel-pt-a_CCSER2_1384-tcr_8_11': True, 

                               'epi_mel-pt-a_GFPT2_896-tcr_18_27': True,
                               'epi_mel-pt-a_ADAM10_1753-tcr_15_24': True,

                               'epi_mel-pt-a_CCSER2_1384-tcr_103_200': True, 
                               'epi_mel-pt-a_ADAM10_1753-tcr_130_255': True, 
                               'epi_mel-pt-a_CCSER2_1384-tcr_170_336': True, 
                               'epi_mel-pt-a_CCSER2_1384-tcr_180_359': True, 
                               'epi_mel-pt-a_CCSER2_1384-tcr_201_401': True, 

                               'epi_mel-pt-a_TNFAIP2_1709-tcr_1_1': True,
                               'epi_mel-pt-a_TNFAIP2_1709-tcr_10_19': True,

                               'epi_mel-pt-a_TNFAIP2_1709-tcr_30_51': True,
                               'epi_mel-pt-a_TNFAIP2_1709-tcr_34_60': True,
                               'epi_mel-pt-a_TNFAIP2_1709-tcr_60_114': True,
                               'epi_mel-pt-a_TNFAIP2_1709-tcr_72_139': True,
                               'epi_mel-pt-a_TNFAIP2_1709-tcr_101_197': True,

                               'epi_mel-pt-a_CCSER2_1384-tcr_128_252': True, 

                                'epi_mel-pt-a_XRCC6_2333-tcr_95_186': False, 
                                'epi_mel-pt-a_FUT8_1672A-tcr_69_132': False,  

                                'epi_mel-pt-a_CCSER2_1384-tcr_169_335': False,
                                'epi_mel-pt-a_TNFAIP2_1709-tcr_164_325': False, 
                                'epi_mel-pt-a_ACAP3_5-tcr_181_362': False, 
                                'epi_mel-pt-a_CBLB_629-tcr_196_392': False, 
                                'epi_mel-pt-a_CBLB_629-tcr_87_170': False, 
                                'epi_mel-pt-a_ZC3H13_1617-tcr_32_55': False,
                                'epi_mel-pt-a_MYH11_1809B-tcr_2_2': False, 
                                'epi_mel-pt-a_KAT6B_1378-tcr_21_34': False,  
                                'epi_mel-pt-a_CASP8_461A-tcr_17_26': False, 
                                'epi_mel-pt-a_NTNG1_111-tcr_5_7': False, 
                                'epi_mel-pt-a_PNPLA8-tcr_87_170': False,
                                'epi_mel-pt-a_LAMA5_2271-tcr_181_362': False              
}

In [7]:
plot_detected_tcr_epi_plate_map(project_dir = project_dir,
                                counts_df_dict = counts_df_dict,
                                validating_pair_dict = validating_pair_dict,
                                epitope_barcode_length = 18,
                                detected_fontsize = 3)

Plotting plate: 1-3
Plotting plate: 1-11
Plotting plate: 1-9
Plotting plate: 1-5
Plotting plate: 1-8
Plotting plate: 1-4
Plotting plate: 1-2
Plotting plate: 1-7
Plotting plate: 1-1
Plotting plate: 1-6
Plotting plate: 1-13
Plotting plate: 1-10
Plotting plate: 1-12


In [ ]:
plates_list_name = 'mel-pt-a-200-vs-200-screen-1'
plates_list = ['1-7', '1-8', '1-9','1-10', '1-11', '1-12', '1-13']


write_tcr_epi_pair_well_count_files(project_dir = project_dir,
                                    plates_list = plates_list,
                                    plates_list_name = plates_list_name,
                                    counts_df_dict = counts_df_dict.copy(),
                                    collapse_epitope_barcode = True,
                                    epitope_barcode_length= 18,
                                    validating_pair_dict = validating_pair_dict
                                   )

Processing plate: 1-7
Processing plate: 1-8
Processing plate: 1-9
Processing plate: 1-10
Processing plate: 1-11
Processing plate: 1-12
Processing plate: 1-13
Writing run-1-2 summary statistics...
Total # of co-culture sort wells counted: 1819


## Bulk seq QC

In [3]:
bulk_project_dir = Path(project_dir) / 'bulk_seq_data' /'TCR'

In [ ]:
name_split_location = 2
for count_file in (bulk_project_dir / "counts").glob("*.csv"):
    print(count_file.name)
    overlap_between_ref_and_count_names(
        count_file=count_file,
        count_file_ref_name_col="reference_name",
        reference_file='/references/T200_Ag200/T200_beta.fa',
        library_ref_id_list=None,
        print_diff_names=True,
    )

    outs_dir = Path(bulk_project_dir) / "outs"
    outs_dir.mkdir(parents=True, exist_ok=True)

    plot_count_histogram(
        count_file=count_file,
        count_file_ref_name_col="reference_name",
        count_file_count_col="read_count",
        bin_min=0,
        bin_max=5,
        bin_size=0.1,
        log10 = True, 
        xlabel="Number of reads per transcript",
        percentile_ratio_filter_threshold = 1.75, 
        title=count_file.stem.split("_")[name_split_location],
        save_file_path=outs_dir / f"{count_file.stem.split('_')[name_split_location]}_count_hist.pdf",
        ylim_max=30,
        w=6.25,
        h=6.25,
    )
    print("\n")

7767_15_1-T200-2-bulk_GGCATTCT-CAAGCTAG_S15_R1_001_trimmed_sorted_2ndfiltered_cigarmd_filtered_counts.csv
# intersecting: 180
# diff: 3
# union: 183
Sensitivity: 98.36% (180/183 reference names found in counts)
refs not in count names: {'143_275', '189_377', '177_349'}
count names not in ref subset: set()
95th percentile: 3230.80 | 5th percentile: 344.00 | 95th/5th ratio: 9.39


7767_14_1-T200-1-bulk_TTACAGGA-GCTTGTCA_S14_R1_001_trimmed_sorted_2ndfiltered_cigarmd_filtered_counts.csv
# intersecting: 180
# diff: 3
# union: 183
Sensitivity: 98.36% (180/183 reference names found in counts)
refs not in count names: {'143_275', '189_377', '177_349'}
count names not in ref subset: set()
95th percentile: 4856.25 | 5th percentile: 686.75 | 95th/5th ratio: 7.07




In [7]:
bulk_project_dir = Path(project_dir) / 'bulk_seq_data' /'Ag'

In [ ]:
name_split_location = 2
for count_file in (bulk_project_dir / "counts").glob("*.csv"):
    print(count_file.name)
    overlap_between_ref_and_count_names(
        count_file=count_file,
        count_file_ref_name_col="reference_name",
        reference_file='/references/T200_Ag200/Ag200.fa',
        library_ref_id_list=None,
        print_diff_names=True,
    )

    outs_dir = Path(bulk_project_dir) / "outs"
    outs_dir.mkdir(parents=True, exist_ok=True)

    plot_count_histogram(
        count_file=count_file,
        count_file_ref_name_col="reference_name",
        count_file_count_col="read_count",
        bin_min=0,
        bin_max=5,
        bin_size=0.1,
        log10 = True,
        xlabel="Number of reads per transcript",
        percentile_ratio_filter_threshold = 1.75, 
        title=count_file.stem.split("_")[name_split_location],
        save_file_path=outs_dir / f"{count_file.stem.split('_')[name_split_location]}_count_hist.pdf",
        ylim_max=30,
        w=6.25,
        h=6.25,
    )
    print("\n")

7767_16_1-Ag200-2-bulk_GACCTGAA-CTCACCAA_S16_R1_001_trimmed_sorted_2ndfiltered_cigarmd_filtered_counts.csv
# intersecting: 400
# diff: 3
# union: 403
Sensitivity: 99.26% (400/403 reference names found in counts)
refs not in count names: {'GLC', 'p20_stuffer', 'NYESO1'}
count names not in ref subset: set()
95th percentile: 2744.90 | 5th percentile: 339.00 | 95th/5th ratio: 8.10




## Mel-Pt-A 200 vs 200 screen 1

In [9]:
epi_bulk_counts_df = read_gdna_counts_csv(count_file = project_dir / 'bulk_seq_data/Ag/counts/7767_16_1-Ag200-2-bulk_GACCTGAA-CTCACCAA_S16_R1_001_trimmed_sorted_2ndfiltered_cigarmd_filtered_counts.csv',
                                          count_file_ref_name_col = 'reference_name',
                                          collapse_epitope_barcode = True,
                                          epitope_barcode_length = 18
                                          )

In [10]:
normalized_epi_bulk_counts_df = normalize_counts_df_by_total_counts(count_df = epi_bulk_counts_df, count_col = 'read_count')

In [11]:
tcr_bulk_counts_df_1 = read_gdna_counts_csv(count_file = project_dir / 'bulk_seq_data/TCR/counts/7767_14_1-T200-1-bulk_TTACAGGA-GCTTGTCA_S14_R1_001_trimmed_sorted_2ndfiltered_cigarmd_filtered_counts.csv',
                                         count_file_ref_name_col = 'reference_name',
                                         )

In [12]:
normalized_tcr_bulk_counts_df_1 = normalize_counts_df_by_total_counts(count_df = tcr_bulk_counts_df_1, count_col = 'read_count')

In [13]:
tcr_bulk_counts_df_2 = read_gdna_counts_csv(count_file = project_dir / 'bulk_seq_data/TCR/counts/7767_15_1-T200-2-bulk_GGCATTCT-CAAGCTAG_S15_R1_001_trimmed_sorted_2ndfiltered_cigarmd_filtered_counts.csv',
                                         count_file_ref_name_col = 'reference_name',
                                         )

In [14]:
normalized_tcr_bulk_counts_df_2 = normalize_counts_df_by_total_counts(count_df = tcr_bulk_counts_df_2, count_col = 'read_count')

In [15]:
print(len(set(normalized_tcr_bulk_counts_df_1.index).union(set(normalized_tcr_bulk_counts_df_2.index))))
print(len(set(normalized_tcr_bulk_counts_df_1.index).intersection(set(normalized_tcr_bulk_counts_df_2.index))))
print(len(set(normalized_tcr_bulk_counts_df_1.index).difference(set(normalized_tcr_bulk_counts_df_2.index))))
normalized_tcr_bulk_counts_df = normalized_tcr_bulk_counts_df_1.merge(normalized_tcr_bulk_counts_df_2, on = 'reference_name', suffixes = ('_1', '_2'))
print('Spearman’s rank correlation coefficient between T200-1 and T200-2:', normalized_tcr_bulk_counts_df['read_count_1'].corr(normalized_tcr_bulk_counts_df['read_count_2'], method = 'spearman'))
normalized_tcr_bulk_counts_df['read_count'] = normalized_tcr_bulk_counts_df.loc[:, ['read_count_1', 'read_count_2']].mean(axis = 1)
normalized_tcr_bulk_counts_df.drop(['read_count_1', 'read_count_2'], axis = 1, inplace = True)
normalized_tcr_bulk_counts_df['read_count'].isna().any()

180
180
0
Spearman’s rank correlation coefficient between T200-1 and T200-2: 0.8152984853895983


np.False_

In [16]:
pair_count_matrix_df = read_tcr_epi_pair_counts_csv(pair_count_matrix_csv = project_dir / 'outs' / plates_list_name / f'pair_count_matrix_df_{plates_list_name}.csv')

In [ ]:
exp_pair_prob_matrix_df, pair_count_matrix_df = calculate_tcr_epi_pair_exp_prob_matrix(pair_count_matrix_df = pair_count_matrix_df,
                                                                                       normalized_epitope_bulk_counts_df = normalized_epi_bulk_counts_df,
                                                                                       normalized_tcr_bulk_counts_df = normalized_tcr_bulk_counts_df,
                                                                                       bulk_epitope_count_col = 'read_count',
                                                                                       bulk_tcr_count_col = 'read_count',
                                                                                       normalize_using_plate = False,
                                                                                       )

In [18]:
alpha = 0.01

In [19]:
pair_fold_changes_df = obs_tcr_epi_count_binom_sf_test(project_dir = project_dir,
                                                       pair_count_matrix_df = pair_count_matrix_df,
                                                       exp_pair_prob_matrix_df = exp_pair_prob_matrix_df,
                                                       plates_list_name = plates_list_name,
                                                       validating_pair_dict = validating_pair_dict,
                                                       fdr_correction = True,
                                                       alpha = alpha,
                                                       normalize_total_called_pairs = False,
                                                       normalize_total_wells_with_detected_pair = True,
                                                       sort_fold_change = False,
                                                       sort_pvalue = True
                                                      )

1668 TCR-epitope pairs have non-zero well counts of 32111 tested TCR-epitope pairs


In [20]:
pair_fold_changes_df = add_tcr_epi_pairs_only_supported_by_co_occurrence_bool_col(project_dir = project_dir,
                                                                                  plates_list_name = plates_list_name,
                                                                                  pair_fold_changes_df = pair_fold_changes_df,
                                                                                  epi_transcript_ratio = 0.7,
                                                                                  tcr_transcript_ratio = 0.7
                                                                                 )

In [21]:
pair_fold_changes_df = add_pair_entropy_over_plate_col(pair_fold_changes_df = pair_fold_changes_df, project_dir = project_dir)

In [23]:
pair_fold_changes_df = add_pair_entropy_over_plate_col(pair_fold_changes_df = pair_fold_changes_df, project_dir = project_dir)

In [24]:
pair_fold_changes_df.to_excel(project_dir / 'outs'/ plates_list_name /f'pair_fold_changes_df_{plates_list_name}.xlsx')
pair_fold_changes_df = pd.read_excel(Path(project_dir) / "outs" / plates_list_name / f"pair_fold_changes_df_{plates_list_name}.xlsx", index_col = 0)

In [25]:
pair_fold_changes_df['Fold_change'].max()

6055.867169595543

In [ ]:
plot_tcr_epi_pair_ma_plot(pair_fold_changes_df = pair_fold_changes_df,
                          title = plates_list_name,
                          xlim_max = 0.0002,
                          xticks_list = list(np.arange(-0.00001, 0.0002, 0.0001)),
                          xlim_min = -0.00001,
                          ylim_max = 1_000,
                          save_dir = project_dir / 'outs'/ plates_list_name,
                          custom_pair_color_dict = None,
                          plot_filter_only_supported_co_occurrence = True,
                          plot_reactive_pair_co_occurrence = False,
                          annotate = True,
                          annotation_fontsize = 2.5,
                          w = 6,
                          h = 6
                         )

In [27]:
plot_tcr_epi_pair_ma_plot(pair_fold_changes_df = pair_fold_changes_df,
                          title = plates_list_name,
                          xlim_max = 0.0002,
                          xticks_list = list(np.arange(-0.00001, 0.0002, 0.0001)),
                          xlim_min = -0.00001,
                          ylim_max = 1_000,
                          save_dir = project_dir / 'outs' / plates_list_name,
                          custom_pair_color_dict = None,
                          plot_filter_only_supported_co_occurrence = True,
                          plot_reactive_pair_co_occurrence = False,
                          annotate = False,
                          #annotation_fontsize = 2.5,
                          w = 6,
                          h = 6
                         )

In [ ]:
plot_pvalue_corrected_rank_vs_pvalue_corrected(pair_fold_changes_df = pair_fold_changes_df,
                                               save_dir = project_dir / 'outs' / plates_list_name,
                                               title = plates_list_name,
                                               xlim_max = 80,
                                               xlim_min = -5,
                                               ylim_max = 65,
                                               ylim_min = -2.5,
                                               adj_pval_alpha = alpha, 
                                               filter_significant_with_more_than_one_epi = True,
                                               custom_pair_color_dict = None,
                                               annotate = True
                                               )

In [ ]:
plot_pvalue_corrected_rank_vs_pvalue_corrected(pair_fold_changes_df = pair_fold_changes_df,
                                               save_dir = project_dir / 'outs' / plates_list_name, 
                                               title = plates_list_name,
                                               xlim_max = 80,
                                               xlim_min = -5,
                                               ylim_max = 65,
                                               ylim_min = -2.5,
                                               adj_pval_alpha = alpha, 
                                               filter_significant_with_more_than_one_epi = True,
                                               custom_pair_color_dict = None,
                                               annotate = False
                                               )

In [30]:
write_reactive_pair_results(pair_fold_changes_df = pair_fold_changes_df,
                            project_dir = project_dir,
                            plates_list_name = plates_list_name,
                            validating_pair_dict = validating_pair_dict
                           )